# Day 6 Tutorial：Gradient Boosting 顺序修错

## Goal

训练固定梯度提升回归基线，并使用 `staged_predict` 观察逐阶段 train/validation RMSE。

## Setup

先用三个数手算一次缩放修正，再使用固定人工数据。

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_demo = np.array([1.0, 2.0, 6.0])
current = np.array([3.0, 3.0, 3.0])
correction = np.array([-1.5, -1.5, 3.0])
updated = current + 0.1 * correction
new_residual = y_demo - updated
print('updated prediction:', updated)
print('new residual:', new_residual)

updated prediction: [2.85 2.85 3.3 ]
new residual: [-1.85 -0.85  2.7 ]


In [2]:
X_train = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
y_train = np.array([0.2, 1.1, 1.9, 3.2, 3.9, 5.1])
X_valid = np.array([[1.5], [3.5], [5.5]])
y_valid = np.array([1.4, 3.6, 5.4])
print('data shapes:', X_train.shape, y_train.shape, X_valid.shape, y_valid.shape)

data shapes: (6, 1) (6,) (3, 1) (3,)


## Steps

固定学习率、树数和深度，随后提取 100 个阶段的预测。

In [3]:
model = GradientBoostingRegressor(
    learning_rate=0.1,
    n_estimators=100,
    max_depth=2,
    random_state=42,
)
model.fit(X_train, y_train)

valid_prediction = model.predict(X_valid)
fixed_scores = {
    'valid_mae': float(mean_absolute_error(y_valid, valid_prediction)),
    'valid_rmse': float(np.sqrt(mean_squared_error(y_valid, valid_prediction))),
    'valid_r2': float(r2_score(y_valid, valid_prediction)),
}
fixed_scores

{'valid_mae': 0.333360834670224,
 'valid_rmse': 0.33667588497703904,
 'valid_r2': 0.9576347232672071}

In [4]:
staged_train = list(model.staged_predict(X_train))
staged_valid = list(model.staged_predict(X_valid))
stage_rows = []

for stage, (train_pred, valid_pred) in enumerate(
    zip(staged_train, staged_valid), start=1
):
    stage_rows.append({
        'stage': stage,
        'train_rmse': float(np.sqrt(mean_squared_error(y_train, train_pred))),
        'valid_rmse': float(np.sqrt(mean_squared_error(y_valid, valid_pred))),
    })

stage_table = pd.DataFrame(stage_rows)
stage_table[stage_table['stage'].isin([1, 5, 10, 25, 50, 100])]

,stage,train_rmse,valid_rmse
0,1,1.511619,1.698453
4,5,1.013346,1.174561
9,10,0.619311,0.782896
24,25,0.141291,0.400890
49,50,0.011974,0.340555
99,100,0.000086,0.336676


## Checks

核对阶段数量、最终阶段与普通预测一致，并检查训练误差总体下降。

In [5]:
assert model.estimators_.shape == (100, 1)
assert len(staged_valid) == 100
assert np.allclose(staged_valid[-1], valid_prediction)
assert stage_table['train_rmse'].iloc[-1] < stage_table['train_rmse'].iloc[0]
assert np.isfinite(stage_table.to_numpy()).all()
print('staged prediction checks passed')

staged prediction checks passed


## Next Steps

完成 `03_exercises.md`。个人副本可保存 `staged_metrics.csv`，但今天不使用 test 选择阶段，也不把本模型称为 XGBoost。